[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/06_xml_modeling.ipynb)

# 教程 6：MJCF 建模指南

用 MuJoCo 的 XML 格式构建自定义机器人和环境。

In [ ]:
!pip install -q mujoco mediapy

In [ ]:
import mujoco
import numpy as np
import mediapy as media

## 刚体-关节-几何体层次结构

MJCF 采用树状结构：
```
mujoco
└── worldbody
    └── body (has joints & geoms)
        └── body (child body)
            └── ...
```

- **刚体（body）** 定义运动学树中的坐标系
- **关节（joint）** 连接刚体（增加自由度）
- **几何体（geom）** 定义碰撞/视觉形状

In [ ]:
# Build a 2-link arm step by step
arm_xml = """
<mujoco model="2link_arm">
  <option gravity="0 0 -9.81"/>
  
  <!-- Defaults reduce repetition -->
  <default>
    <joint damping="0.5"/>
    <geom rgba="0.3 0.6 0.8 1"/>
  </default>
  
  <worldbody>
    <light pos="0 -1 2" dir="0 0.5 -1"/>
    <geom type="plane" size="1 1 0.1" rgba=".9 .9 .9 1"/>
    
    <!-- Shoulder (fixed to world at height 1) -->
    <body name="upper_arm" pos="0 0 1">
      <joint name="shoulder" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0.4 0 0" size="0.03"/>
      
      <!-- Elbow -->
      <body name="forearm" pos="0.4 0 0">
        <joint name="elbow" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0.3 0 0" size="0.025" rgba="0.8 0.4 0.2 1"/>
        
        <!-- End-effector marker -->
        <site name="tip" pos="0.3 0 0" size="0.02" rgba="1 0 0 1"/>
      </body>
    </body>
  </worldbody>
  
  <actuator>
    <motor joint="shoulder" ctrlrange="-5 5"/>
    <motor joint="elbow" ctrlrange="-3 3"/>
  </actuator>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(arm_xml)
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

renderer = mujoco.Renderer(model, height=400, width=500)
renderer.update_scene(data)
media.show_image(renderer.render())

## 使用默认值

`<default>` 元素让你只需设置一次常用属性。
你也可以为不同的分组使用**默认类（default classes）**：

In [ ]:
xml_defaults = """
<mujoco>
  <default>
    <default class="visual">
      <geom contype="0" conaffinity="0" group="1"/>
    </default>
    <default class="collision">
      <geom group="0" rgba="1 0 0 0.3"/>
    </default>
  </default>
  
  <worldbody>
    <light pos="0 0 2"/>
    <body pos="0 0 1">
      <joint type="free"/>
      <!-- Visual geom (detailed mesh appearance) -->
      <geom class="visual" type="box" size="0.2 0.1 0.05" rgba="0.2 0.5 0.8 1"/>
      <!-- Collision geom (simplified shape) -->
      <geom class="collision" type="box" size="0.22 0.12 0.07"/>
    </body>
  </worldbody>
</mujoco>
"""
m = mujoco.MjModel.from_xml_string(xml_defaults)
print(f"Geoms: {m.ngeom} (visual + collision separated via classes)")

## 复合对象

MuJoCo 可以自动生成软体、绳索和网格：

In [ ]:
rope_xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 2"/>
    <geom type="plane" size="2 2 0.1" rgba=".9 .9 .9 1"/>
    
    <composite type="rope" count="15 1 1" spacing="0.04" offset="0 0 1">
      <joint kind="main" damping="0.01"/>
      <geom type="capsule" size=".01 .015" rgba="0.8 0.2 0.1 1"/>
    </composite>
  </worldbody>
</mujoco>
"""

model_r = mujoco.MjModel.from_xml_string(rope_xml)
data_r = mujoco.MjData(model_r)
renderer_r = mujoco.Renderer(model_r, height=400, width=500)

frames = []
while data_r.time < 3.0:
    mujoco.mj_step(model_r, data_r)
    if len(frames) < data_r.time * 30:
        renderer_r.update_scene(data_r)
        frames.append(renderer_r.render().copy())

renderer_r.close()
media.show_video(frames, fps=30)

## 加载外部模型

使用 `<include>` 组合模型，或从 [MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie) 加载：

In [ ]:
# Example: programmatic model editing (MuJoCo 3.x spec API)
spec = mujoco.MjSpec()
spec.worldbody.add_light(pos=[0, 0, 2])
spec.worldbody.add_geom(type=mujoco.mjtGeom.mjGEOM_PLANE, size=[1, 1, 0.1])

body = spec.worldbody.add_body(name='dynamic_box', pos=[0, 0, 1])
body.add_joint(type=mujoco.mjtJoint.mjJNT_FREE)
body.add_geom(type=mujoco.mjtGeom.mjGEOM_BOX, size=[0.1, 0.1, 0.1], mass=1.0, rgba=[0, 0.7, 0, 1])

model_spec = spec.compile()
data_spec = mujoco.MjData(model_spec)
mujoco.mj_forward(model_spec, data_spec)

renderer_s = mujoco.Renderer(model_spec, height=300, width=400)
renderer_s.update_scene(data_spec)
media.show_image(renderer_s.render())
renderer_s.close()
print("Model created programmatically with MjSpec!")

## 肌腱（tendon）

肌腱是连接关节或空间点的长度约束。
它们可以实现**关节耦合**、**滑轮**和**肌肉走线**。

类型：
- `<fixed>` —— 关节位置的加权和（耦合）
- `<spatial>` —— 走线经过途经点和缠绕表面

In [ ]:
xml_tendon = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 2"/>
    <geom type="plane" size="1 1 0.1" rgba=".9 .9 .9 1"/>
    
    <!-- Two joints coupled by a tendon (like a differential) -->
    <body pos="-0.2 0 0.5">
      <joint name="left" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.3" size="0.02" rgba="0.8 0.2 0.2 1"/>
    </body>
    <body pos="0.2 0 0.5">
      <joint name="right" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.3" size="0.02" rgba="0.2 0.2 0.8 1"/>
    </body>
  </worldbody>
  
  <!-- Fixed tendon: couples left and right joints -->
  <!-- coef=1 for left, coef=-1 for right means they move opposite -->
  <tendon>
    <fixed name="coupled" limited="true" range="-0.5 0.5">
      <joint joint="left" coef="1"/>
      <joint joint="right" coef="-1"/>
    </fixed>
  </tendon>
  
  <actuator>
    <motor joint="left" ctrlrange="-5 5"/>
  </actuator>
</mujoco>
"""

model_t = mujoco.MjModel.from_xml_string(xml_tendon)
data_t = mujoco.MjData(model_t)

# Drive left joint — right joint follows (opposite direction)
times, left_angles, right_angles = [], [], []
while data_t.time < 4.0:
    data_t.ctrl[0] = 3.0 * np.sin(2 * data_t.time)
    mujoco.mj_step(model_t, data_t)
    times.append(data_t.time)
    left_angles.append(np.degrees(data_t.qpos[0]))
    right_angles.append(np.degrees(data_t.qpos[1]))

plt.figure(figsize=(10, 3))
plt.plot(times, left_angles, label='Left joint')
plt.plot(times, right_angles, label='Right joint (coupled)')
plt.xlabel('Time (s)'); plt.ylabel('Angle (deg)')
plt.title('Tendon-Coupled Joints (differential)'); plt.legend(); plt.grid(True); plt.show()
print('The tendon forces the joints to move in opposite directions!')

## 等式约束

等式约束强制几何关系成立：

| 类型 | 效果 |
|------|--------|
| `<connect>` | 两个刚体共享一个点（没有刚体的球关节） |
| `<weld>` | 两个刚体刚性连接 |
| `<joint>` | 一个关节跟随另一个关节（齿轮比） |
| `<tendon>` | 肌腱长度保持固定 |

这些是软约束（以阻抗方式求解），由 `solref` 和 `solimp` 控制。

In [ ]:
xml_equality = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="1 1 0.1" rgba=".9 .9 .9 1"/>
    
    <!-- Master joint (actuated) -->
    <body name="master" pos="-0.3 0 0.5">
      <joint name="j_master" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.3" size="0.025" rgba="0.2 0.7 0.3 1"/>
    </body>
    
    <!-- Follower joint (2:1 gear ratio) -->
    <body name="follower" pos="0.3 0 0.5">
      <joint name="j_follower" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.2" size="0.02" rgba="0.7 0.3 0.2 1"/>
    </body>
  </worldbody>
  
  <!-- Follower moves at 2x the speed of master -->
  <equality>
    <joint joint1="j_master" joint2="j_follower" polycoef="0 2 0 0 0"/>
  </equality>
  
  <actuator>
    <motor joint="j_master" ctrlrange="-5 5"/>
  </actuator>
</mujoco>
"""

model_eq = mujoco.MjModel.from_xml_string(xml_equality)
data_eq = mujoco.MjData(model_eq)

times, master_angles, follower_angles = [], [], []
while data_eq.time < 4.0:
    data_eq.ctrl[0] = 3.0 * np.sin(data_eq.time)
    mujoco.mj_step(model_eq, data_eq)
    times.append(data_eq.time)
    master_angles.append(np.degrees(data_eq.qpos[0]))
    follower_angles.append(np.degrees(data_eq.qpos[1]))

plt.figure(figsize=(10, 3))
plt.plot(times, master_angles, label='Master')
plt.plot(times, follower_angles, label='Follower (2:1 ratio)')
plt.xlabel('Time (s)'); plt.ylabel('Angle (deg)')
plt.title('Equality Constraint: Gear Ratio'); plt.legend(); plt.grid(True); plt.show()
print('Follower moves at 2x the master angle — like a gear train!')